# Motivation

Local LLMs are interesting as they can be used to process private documents (images, PDFs, etc.) and build applications that can fetch relevant information from them. On my M1 MacBook Air with 8GB RAM, I want to experiment with a small LLM that can run in the background without hindering my usage of other applications. With that in mind, in this notebook, I will explore the Qwen3.5 0.8B parameters model. The choice of the model is only partially deliberate. I opted for the 0.8B parameter model as I observed the 2B parameter model, which is the next bigger model from the Qwen3.5 family, sporadically freezed my MacBook in another experiment. I chose Qwen3.5 only because it was reviewed positively by commentators on the internet when it released in March 2026.


In [2]:
from IPython.display import Markdown

from ablation_core import MODEL_ID, QUICK_QUERIES, QUERIES, STAGES, agent_once, answer, load_model, score, table, weather


## Load the MLX model

`mlx-lm` returns a model and tokenizer. The helper functions use the tokenizer's chat template with thinking disabled.


In [3]:
model, tokenizer = load_model()
MODEL_ID


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

'mlx-community/Qwen3.5-0.8B-OptiQ-4bit'

## Run one plain query

Before adding tools, check that the model can answer a normal chat prompt through MLX.


In [3]:
messages = [{"role": "user", "content": "What is 2+2?"}]
answer(model, tokenizer, messages)


'2 + 2 = **4**.\n\nThis is a fundamental concept in mathematics known as the **Addition Table**. It is one of the most basic examples of how numbers work together to create new numbers.'

## Add a simple weather tool

The tool is just `wttr.in` JSON wrapped into a short text string. The model does not call it directly; the agent loop parses a trigger and calls the function.


In [4]:
weather("Berlin")


'Condition: Partly cloudy, Temperature: 23C, Max today: 25C, Min today: 13C, Humidity: 50%'

## Start with the base prompt

The base prompt uses `[WEATHER: city]` and frames the model as a weather assistant. This is the true baseline before the later prompt changes.


In [5]:
prompt, regex = STAGES["base"]
agent_once(model, tokenizer, prompt, regex, "What is the weather in London?")


{'response': '[WEATHER: London is currently experiencing pleasant weather with a high of 22°C and a low of 15°C, accompanied by light winds and a clear sky.]',
 'city': None}

## Show a failure mode

The base prompt can over-apply the weather trigger. A non-weather question should not request the tool.


In [6]:
prompt, regex = STAGES["base"]
agent_once(model, tokenizer, prompt, regex, "What is the capital of France?")


{'response': '[WEATHER: Paris]', 'city': 'Paris'}

## Evaluate additive stages

Now add prompt changes cumulatively. The notebook defaults to four quick queries and one run each; use `QUERIES` and `N_RUNS = 5` for the full result.


In [7]:
eval_queries = QUICK_QUERIES
N_RUNS = 1
(len(eval_queries), N_RUNS)


(4, 1)

In [8]:
results = {
    name: score(model, tokenizer, stage, eval_queries, N_RUNS)
    for name, stage in STAGES.items()
}


In [9]:
Markdown(table(results))


| Stage | Non-weather | Weather | Total |
|---|---:|---:|---:|
| base | 1/2 | 1/2 | 2/4 |
| + framing | 1/2 | 1/2 | 2/4 |
| + trigger | 1/2 | 1/2 | 2/4 |
| + gate | 1/2 | 2/2 | 3/4 |
| + anti-loop | 1/2 | 2/2 | 3/4 |

## Run the full eval when needed

For evidence, rerun the previous two cells after setting `eval_queries = QUERIES` and `N_RUNS = 5`. That gives 12 queries × 5 runs = 60 trials per stage.
